In [ ]:
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
from qiskit.circuit.library import Initialize
from qiskit import quantum_info as qi
from qiskit.visualization import array_to_latex, plot_histogram
from qiskit.compiler import transpile
from qiskit_aer import Aer
from qiskit.primitives import StatevectorSampler
import numpy as np

#### Function evaluation $~f : \{0, 1 \}^{n} \rightarrow \{ 0, 1 \}$

<div>
<img src="images/function-eval.png" width="300"/>
</div>

#### Source: 

`https://github.com/qiskit-community/qiskit-community-tutorials/tree/master/algorithms/deutsch_josza.ipynb`

#### The dimension of the domain of the function to query

In [ ]:
n = 6
N = 2**n

#### The oracle

$\bullet~$ randomly generate the type of the function to be "oracled"

In [ ]:
oracleType, oracleValue = np.random.randint(2), np.random.randint(2)

if oracleType == 0:
    print("\nThe oracle returns a constant value: ", oracleValue,"\n")
else:
    print("\nThe oracle returns a balanced function\n")
    a = np.random.randint(1,2**n) # this is a hidden parameter for balanced oracle. 

#### Run with all $2^{n}$ arguments of the function to be queried

In [ ]:
backend = Aer.get_backend('qasm_simulator')
count_0 = 0
count_1 = 0
for k in range(N):
    qr_O = QuantumRegister(n+1)
    cr_O = ClassicalRegister(1)
    qc_O = QuantumCircuit(qr_O, cr_O)
    psi = qi.Statevector.from_int(k, dims=N)
    init_gate = Initialize(psi)
    init_gate.label = "init"
    qc_O.append(init_gate, qr_O[:n])
    if oracleType == 0: ## If the oracleType is "0", the oracle returns oracleValue for all input. 
        if oracleValue == 1:
            qc_O.x(qr_O[n])
        else:
            qc_O.id(qr_O[n])
    else: ## Otherwise, it returns the inner product of the input with a (non-zero bitstring) 
        for i in range(n):
            if (a & (1 << i)):
                qc_O.cx(qr_O[i], qr_O[n])   
    qc_O.measure(qr_O[n], cr_O)
    job = backend.run(transpile(qc_O, backend), shots=1)
    counts = job.result().get_counts()
    if (counts.get("0") == 1):
        count_0 += 1
    if (counts.get("1") == 1):
        count_1 += 1
print("f(x)=0: ", count_0, " times, f(x)=1: ", count_1, " times")

#### With this oracle, build the circuit for the D-J algorithm

In [ ]:
qr = QuantumRegister(n+1)
cr = ClassicalRegister(n)
qc = QuantumCircuit(qr, cr)

#### Treat all function arguments at once in a superposition: apply Hadamard to the $n$-register

#### $\bullet~$ check that we can write the result like this:

#### $H^{\otimes n} = \frac{1}{\sqrt{2^{n}}} \sum_{y=0}^{2^{n}-1} (-1)^{x \cdot y} \lvert y \rangle$

#### $x \cdot y = x_{n-1}y_{n-1} \oplus x_{n-2}y_{n-2} \oplus \ldots \oplus x_{0}y_{0}$

In [ ]:
qc.h(qr[0:n])

#### Prepare the ancilla qubit

In [ ]:
qc.x(qr[n])
qc.h(qr[n])
qc.barrier()

In [ ]:
if oracleType == 0:#If the oracleType is "0", the oracle returns oracleValue for all input. 
    if oracleValue == 1:
        qc.x(qr[n])
    else:
        qc.id(qr[n])
else: # Otherwise, it returns the inner product of the input with a (non-zero bitstring) 
    for i in range(n):
        if (a & (1 << i)):
            qc.cx(qr[i], qr[n])

In [ ]:
# Apply barrier to mark the end of the oracle
qc.barrier()

# Apply Hadamard gates after querying the oracle
for i in range(n):
    qc.h(qr[i])
    
# Measurement
qc.barrier()
for i in range(n):
    qc.measure(qr[i], cr[i])

In [ ]:
qc.draw(output='mpl',scale=0.6)

In [ ]:
backend = Aer.get_backend('qasm_simulator')
job = backend.run(transpile(qc, backend), shots=1)
results = job.result()
counts = results.get_counts()
print(counts)

In [ ]:
if oracleType == 0:
    print("\nThe oracle returns a constant value: ", oracleValue,".\n")
else:
    print("\nThe oracle returns a balanced function.\n")